# Complete Model Evaluation - CNN, PPO, and DQN

This notebook evaluates ALL THREE models on 200 test images and computes:
- Overall IoU
- Thin Cloud IoU
- All performance metrics (Accuracy, Precision, Recall, F1)

**Run this in Google Colab with GPU runtime.**

## 1. Setup and Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install stable-baselines3 gymnasium rasterio scikit-learn -q

print("✅ Setup complete!")

## 2. Configuration and Imports

In [ ]:
import os
import glob
import numpy as np
import rasterio
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score, accuracy_score
from stable_baselines3 import DQN, PPO
import gymnasium as gym
from gymnasium import spaces

# Paths - UPDATE THESE IF NEEDED
DATA_DIR = '/content/drive/MyDrive/Colab_Data/cloudsen12_processed_1000'
DQN_MODEL_PATH = '/content/drive/MyDrive/Colab_Data/dqn_thin_cloud/dqn_thin_cloud_100000_steps.zip'
PPO_MODEL_PATH = '/content/drive/MyDrive/Colab_Data/ppo_thin_cloud/ppo_thin_cloud_720000_steps.zip'

# Verify paths exist
print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
print(f"DQN model exists: {os.path.exists(DQN_MODEL_PATH)}")
print(f"PPO model exists: {os.path.exists(PPO_MODEL_PATH)}")

# Count files
image_files = sorted(glob.glob(f'{DATA_DIR}/*_image.tif'))
mask_files = sorted(glob.glob(f'{DATA_DIR}/*_mask.tif'))
print(f"\n📂 Found {len(image_files)} images and {len(mask_files)} masks")

## 3. Define Environments (DQN Discrete + PPO Continuous)

In [ ]:
class ThinCloudDetectionEnvDiscrete(gym.Env):
    """
    Discrete action space environment for DQN thin cloud detection.
    15 discrete actions: combinations of threshold adjustments and boosts.
    """
    
    def __init__(self, cnn_prob, ground_truth, patch_size=64):
        super().__init__()
        
        self.cnn_prob = cnn_prob.astype(np.float32)
        self.ground_truth = ground_truth
        self.patch_size = patch_size
        self.h, self.w = cnn_prob.shape
        
        # Create thin cloud mask (class 2)
        self.thin_cloud_mask = (ground_truth == 2)
        self.cloud_mask = (ground_truth >= 1)  # All clouds
        
        # Grid of patches
        self.n_patches_h = self.h // patch_size
        self.n_patches_w = self.w // patch_size
        self.total_patches = self.n_patches_h * self.n_patches_w
        
        # 15 discrete actions: 5 thresholds × 3 boosts
        self.action_space = spaces.Discrete(15)
        
        # Action mapping
        self.threshold_values = [-0.20, -0.10, 0.00, 0.10, 0.20]
        self.boost_values = [0.00, 0.25, 0.50]
        
        # 20-dim observation space
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32
        )
        
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        
    def _get_action_values(self, action):
        """Convert discrete action to threshold and boost values."""
        thresh_idx = action // 3
        boost_idx = action % 3
        return self.threshold_values[thresh_idx], self.boost_values[boost_idx]
    
    def _get_patch_coords(self, patch_idx):
        """Get patch coordinates."""
        row = patch_idx // self.n_patches_w
        col = patch_idx % self.n_patches_w
        y1 = row * self.patch_size
        y2 = y1 + self.patch_size
        x1 = col * self.patch_size
        x2 = x1 + self.patch_size
        return y1, y2, x1, x2
    
    def _get_observation(self):
        """Extract 20-feature observation for current patch."""
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        patch_prob = self.cnn_prob[y1:y2, x1:x2]
        patch_thin = self.thin_cloud_mask[y1:y2, x1:x2]
        
        # CNN probability statistics
        prob_mean = np.mean(patch_prob)
        prob_std = np.std(patch_prob)
        prob_max = np.max(patch_prob)
        prob_min = np.min(patch_prob)
        
        # Probability distribution
        prob_median = np.median(patch_prob)
        prob_q25 = np.percentile(patch_prob, 25)
        prob_q75 = np.percentile(patch_prob, 75)
        
        # Edge/gradient features
        grad_y = np.abs(np.diff(patch_prob, axis=0)).mean()
        grad_x = np.abs(np.diff(patch_prob, axis=1)).mean()
        
        # Thin cloud indicators
        thin_ratio = np.mean(patch_thin)
        uncertain_ratio = np.mean((patch_prob > 0.3) & (patch_prob < 0.7))
        
        # Spatial context
        row_norm = (self.current_patch // self.n_patches_w) / self.n_patches_h
        col_norm = (self.current_patch % self.n_patches_w) / self.n_patches_w
        
        # High probability region
        high_prob_ratio = np.mean(patch_prob > 0.5)
        low_prob_ratio = np.mean(patch_prob < 0.3)
        
        # Texture features
        local_var = np.var(patch_prob)
        
        # Additional features
        prob_range = prob_max - prob_min
        skewness = ((patch_prob - prob_mean) ** 3).mean() / (prob_std ** 3 + 1e-8)
        
        obs = np.array([
            prob_mean, prob_std, prob_max, prob_min,
            prob_median, prob_q25, prob_q75,
            grad_y, grad_x,
            thin_ratio, uncertain_ratio,
            row_norm, col_norm,
            high_prob_ratio, low_prob_ratio,
            local_var, prob_range, skewness,
            0.0, 0.0  # Padding to 20 features
        ], dtype=np.float32)
        
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        return self._get_observation(), {}
    
    def step(self, action):
        threshold_delta, thin_boost = self._get_action_values(action)
        
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        # Apply refinement
        patch = self.refined_prob[y1:y2, x1:x2].copy()
        
        # Lower threshold = more sensitive (add negative delta to prob)
        patch = patch - threshold_delta
        
        # Apply thin cloud boost to uncertain regions
        uncertain_mask = (patch > 0.2) & (patch < 0.6)
        patch[uncertain_mask] += thin_boost
        
        patch = np.clip(patch, 0, 1)
        self.refined_prob[y1:y2, x1:x2] = patch
        
        # Move to next patch
        self.current_patch += 1
        done = self.current_patch >= self.total_patches
        
        # Compute reward
        reward = self._compute_reward(y1, y2, x1, x2)
        
        if done:
            obs = np.zeros(20, dtype=np.float32)
        else:
            obs = self._get_observation()
        
        return obs, reward, done, False, {}
    
    def _compute_reward(self, y1, y2, x1, x2):
        """Multi-objective reward: 70% thin cloud IoU + 30% F1."""
        patch_pred = (self.refined_prob[y1:y2, x1:x2] > 0.5).flatten()
        patch_gt_cloud = self.cloud_mask[y1:y2, x1:x2].flatten()
        patch_gt_thin = self.thin_cloud_mask[y1:y2, x1:x2].flatten()
        
        # Thin cloud IoU
        thin_intersection = np.sum(patch_pred & patch_gt_thin)
        thin_union = np.sum(patch_pred | patch_gt_thin)
        if thin_union > 0:
            thin_iou = thin_intersection / (thin_union + 1e-8)
        else:
            thin_iou = 0.5
        
        # F1 score
        tp = np.sum(patch_pred & patch_gt_cloud)
        fp = np.sum(patch_pred & ~patch_gt_cloud)
        fn = np.sum(~patch_pred & patch_gt_cloud)
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
        reward = 0.7 * thin_iou + 0.3 * f1
        return reward
    
    def get_refined_mask(self):
        """Return the refined binary mask."""
        return (self.refined_prob > 0.5).astype(np.uint8)


class ThinCloudDetectionEnvContinuous(gym.Env):
    """
    Continuous action space environment for PPO thin cloud detection.
    Actions: [threshold_delta, thin_cloud_boost]
    """
    
    def __init__(self, cnn_prob, ground_truth, patch_size=64):
        super().__init__()
        
        self.cnn_prob = cnn_prob.astype(np.float32)
        self.ground_truth = ground_truth
        self.patch_size = patch_size
        self.h, self.w = cnn_prob.shape
        
        # Create thin cloud mask (class 2)
        self.thin_cloud_mask = (ground_truth == 2)
        self.cloud_mask = (ground_truth >= 1)  # All clouds
        
        # Grid of patches
        self.n_patches_h = self.h // patch_size
        self.n_patches_w = self.w // patch_size
        self.total_patches = self.n_patches_h * self.n_patches_w
        
        # Continuous action space: [threshold_delta, thin_boost]
        self.action_space = spaces.Box(
            low=np.array([-0.3, 0.0]),
            high=np.array([0.3, 0.5]),
            dtype=np.float32
        )
        
        # 20-dim observation space
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32
        )
        
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        
    def _get_patch_coords(self, patch_idx):
        """Get patch coordinates."""
        row = patch_idx // self.n_patches_w
        col = patch_idx % self.n_patches_w
        y1 = row * self.patch_size
        y2 = y1 + self.patch_size
        x1 = col * self.patch_size
        x2 = x1 + self.patch_size
        return y1, y2, x1, x2
    
    def _get_observation(self):
        """Extract 20-feature observation for current patch."""
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        patch_prob = self.cnn_prob[y1:y2, x1:x2]
        patch_thin = self.thin_cloud_mask[y1:y2, x1:x2]
        
        # CNN probability statistics
        prob_mean = np.mean(patch_prob)
        prob_std = np.std(patch_prob)
        prob_max = np.max(patch_prob)
        prob_min = np.min(patch_prob)
        
        # Probability distribution
        prob_median = np.median(patch_prob)
        prob_q25 = np.percentile(patch_prob, 25)
        prob_q75 = np.percentile(patch_prob, 75)
        
        # Edge/gradient features
        grad_y = np.abs(np.diff(patch_prob, axis=0)).mean()
        grad_x = np.abs(np.diff(patch_prob, axis=1)).mean()
        
        # Thin cloud indicators
        thin_ratio = np.mean(patch_thin)
        uncertain_ratio = np.mean((patch_prob > 0.3) & (patch_prob < 0.7))
        
        # Spatial context
        row_norm = (self.current_patch // self.n_patches_w) / self.n_patches_h
        col_norm = (self.current_patch % self.n_patches_w) / self.n_patches_w
        
        # High probability region
        high_prob_ratio = np.mean(patch_prob > 0.5)
        low_prob_ratio = np.mean(patch_prob < 0.3)
        
        # Texture features
        local_var = np.var(patch_prob)
        
        # Additional features
        prob_range = prob_max - prob_min
        skewness = ((patch_prob - prob_mean) ** 3).mean() / (prob_std ** 3 + 1e-8)
        
        obs = np.array([
            prob_mean, prob_std, prob_max, prob_min,
            prob_median, prob_q25, prob_q75,
            grad_y, grad_x,
            thin_ratio, uncertain_ratio,
            row_norm, col_norm,
            high_prob_ratio, low_prob_ratio,
            local_var, prob_range, skewness,
            0.0, 0.0  # Padding to 20 features
        ], dtype=np.float32)
        
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        return self._get_observation(), {}
    
    def step(self, action):
        threshold_delta = float(action[0])
        thin_boost = float(action[1])
        
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        # Apply refinement
        patch = self.refined_prob[y1:y2, x1:x2].copy()
        
        # Lower threshold = more sensitive (add negative delta to prob)
        patch = patch - threshold_delta
        
        # Apply thin cloud boost to uncertain regions
        uncertain_mask = (patch > 0.2) & (patch < 0.6)
        patch[uncertain_mask] += thin_boost
        
        patch = np.clip(patch, 0, 1)
        self.refined_prob[y1:y2, x1:x2] = patch
        
        # Move to next patch
        self.current_patch += 1
        done = self.current_patch >= self.total_patches
        
        # Compute reward
        reward = self._compute_reward(y1, y2, x1, x2)
        
        if done:
            obs = np.zeros(20, dtype=np.float32)
        else:
            obs = self._get_observation()
        
        return obs, reward, done, False, {}
    
    def _compute_reward(self, y1, y2, x1, x2):
        """Multi-objective reward: 70% thin cloud IoU + 30% F1."""
        patch_pred = (self.refined_prob[y1:y2, x1:x2] > 0.5).flatten()
        patch_gt_cloud = self.cloud_mask[y1:y2, x1:x2].flatten()
        patch_gt_thin = self.thin_cloud_mask[y1:y2, x1:x2].flatten()
        
        # Thin cloud IoU
        thin_intersection = np.sum(patch_pred & patch_gt_thin)
        thin_union = np.sum(patch_pred | patch_gt_thin)
        if thin_union > 0:
            thin_iou = thin_intersection / (thin_union + 1e-8)
        else:
            thin_iou = 0.5
        
        # F1 score
        tp = np.sum(patch_pred & patch_gt_cloud)
        fp = np.sum(patch_pred & ~patch_gt_cloud)
        fn = np.sum(~patch_pred & patch_gt_cloud)
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
        reward = 0.7 * thin_iou + 0.3 * f1
        return reward
    
    def get_refined_mask(self):
        """Return the refined binary mask."""
        return (self.refined_prob > 0.5).astype(np.uint8)


print("✅ Both environments defined (Discrete for DQN, Continuous for PPO)!")

## 4. Load Models and Test Data

In [ ]:
# Load DQN model
print("Loading DQN model...")
dqn_model = DQN.load(DQN_MODEL_PATH)
print("✅ DQN model loaded!")

# Load PPO model
print("Loading PPO model...")
ppo_model = PPO.load(PPO_MODEL_PATH)
print("✅ PPO model loaded!")

# Load test data (last 200 images = 20% of 1000)
TRAIN_SPLIT = 0.8
split_idx = int(TRAIN_SPLIT * len(image_files))

test_images = image_files[split_idx:]
test_masks = mask_files[split_idx:]

print(f"\n📊 Test set: {len(test_images)} images")

## 5. CNN Baseline Function

In [ ]:
# Try to use s2cloudless, fallback to simple threshold
try:
    from s2cloudless import S2PixelCloudDetector
    cloud_detector = S2PixelCloudDetector(threshold=0.4, average_over=4, dilation_size=2, all_bands=True)
    USE_S2CLOUDLESS = True
    print("✅ Using s2cloudless for baseline (all_bands=True)")
except ImportError:
    !pip install s2cloudless -q
    from s2cloudless import S2PixelCloudDetector
    cloud_detector = S2PixelCloudDetector(threshold=0.4, average_over=4, dilation_size=2, all_bands=True)
    USE_S2CLOUDLESS = True
    print("✅ Installed and using s2cloudless for baseline (all_bands=True)")

def get_cnn_probability(image_path):
    """Get CNN cloud probability map."""
    with rasterio.open(image_path) as src:
        bands = src.read()  # Shape: (13, H, W)
    
    # Normalize bands to 0-1
    bands = bands.astype(np.float32) / 10000.0
    bands = np.clip(bands, 0, 1)
    
    # Reshape for s2cloudless: (1, H, W, 13)
    bands_reshaped = np.transpose(bands, (1, 2, 0))[np.newaxis, ...]
    
    # Get probability
    prob = cloud_detector.get_cloud_probability_maps(bands_reshaped)[0]
    
    return prob.astype(np.float32)

print("✅ CNN baseline function ready!")

## 6. Evaluate All Models on Test Set

In [ ]:
# Metrics accumulators for all three models
baseline_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
ppo_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
dqn_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}

# Thin cloud specific
baseline_thin = {'tp': 0, 'total': 0}
ppo_thin = {'tp': 0, 'total': 0}
dqn_thin = {'tp': 0, 'total': 0}

# IoU accumulators
baseline_iou_sum = 0
ppo_iou_sum = 0
dqn_iou_sum = 0

baseline_thin_iou_sum = 0
ppo_thin_iou_sum = 0
dqn_thin_iou_sum = 0

n_images = 0

print("Evaluating all models on test set...")
print("="*60)

for i, (img_path, mask_path) in enumerate(zip(test_images, test_masks)):
    if (i + 1) % 20 == 0:
        print(f"Processing image {i+1}/{len(test_images)}...")
    
    try:
        # Load ground truth
        with rasterio.open(mask_path) as src:
            gt = src.read(1)
        
        # Get CNN probability
        cnn_prob = get_cnn_probability(img_path)
        
        # Baseline prediction (threshold 0.5)
        baseline_pred = (cnn_prob > 0.5).astype(np.uint8)
        
        # ========== DQN Refinement ==========
        env_dqn = ThinCloudDetectionEnvDiscrete(cnn_prob, gt)
        obs, _ = env_dqn.reset()
        done = False
        while not done:
            action, _ = dqn_model.predict(obs, deterministic=True)
            obs, _, done, _, _ = env_dqn.step(action)
        dqn_pred = env_dqn.get_refined_mask()
        
        # ========== PPO Refinement ==========
        env_ppo = ThinCloudDetectionEnvContinuous(cnn_prob, gt)
        obs, _ = env_ppo.reset()
        done = False
        while not done:
            action, _ = ppo_model.predict(obs, deterministic=True)
            obs, _, done, _, _ = env_ppo.step(action)
        ppo_pred = env_ppo.get_refined_mask()
        
        # Ground truth masks
        gt_cloud = (gt >= 1)  # All clouds
        gt_thin = (gt == 2)   # Thin clouds only
        
        # Flatten for metrics
        baseline_flat = baseline_pred.flatten().astype(bool)
        ppo_flat = ppo_pred.flatten().astype(bool)
        dqn_flat = dqn_pred.flatten().astype(bool)
        gt_cloud_flat = gt_cloud.flatten()
        gt_thin_flat = gt_thin.flatten()
        
        # ========== Overall metrics ==========
        # Baseline
        baseline_metrics['tp'] += np.sum(baseline_flat & gt_cloud_flat)
        baseline_metrics['fp'] += np.sum(baseline_flat & ~gt_cloud_flat)
        baseline_metrics['tn'] += np.sum(~baseline_flat & ~gt_cloud_flat)
        baseline_metrics['fn'] += np.sum(~baseline_flat & gt_cloud_flat)
        
        # PPO
        ppo_metrics['tp'] += np.sum(ppo_flat & gt_cloud_flat)
        ppo_metrics['fp'] += np.sum(ppo_flat & ~gt_cloud_flat)
        ppo_metrics['tn'] += np.sum(~ppo_flat & ~gt_cloud_flat)
        ppo_metrics['fn'] += np.sum(~ppo_flat & gt_cloud_flat)
        
        # DQN
        dqn_metrics['tp'] += np.sum(dqn_flat & gt_cloud_flat)
        dqn_metrics['fp'] += np.sum(dqn_flat & ~gt_cloud_flat)
        dqn_metrics['tn'] += np.sum(~dqn_flat & ~gt_cloud_flat)
        dqn_metrics['fn'] += np.sum(~dqn_flat & gt_cloud_flat)
        
        # ========== Thin cloud recall ==========
        if np.sum(gt_thin_flat) > 0:
            baseline_thin['tp'] += np.sum(baseline_flat & gt_thin_flat)
            baseline_thin['total'] += np.sum(gt_thin_flat)
            ppo_thin['tp'] += np.sum(ppo_flat & gt_thin_flat)
            ppo_thin['total'] += np.sum(gt_thin_flat)
            dqn_thin['tp'] += np.sum(dqn_flat & gt_thin_flat)
            dqn_thin['total'] += np.sum(gt_thin_flat)
        
        # ========== IoU calculation ==========
        # Baseline
        baseline_intersection = np.sum(baseline_flat & gt_cloud_flat)
        baseline_union = np.sum(baseline_flat | gt_cloud_flat)
        if baseline_union > 0:
            baseline_iou_sum += baseline_intersection / baseline_union
        
        # PPO
        ppo_intersection = np.sum(ppo_flat & gt_cloud_flat)
        ppo_union = np.sum(ppo_flat | gt_cloud_flat)
        if ppo_union > 0:
            ppo_iou_sum += ppo_intersection / ppo_union
        
        # DQN
        dqn_intersection = np.sum(dqn_flat & gt_cloud_flat)
        dqn_union = np.sum(dqn_flat | gt_cloud_flat)
        if dqn_union > 0:
            dqn_iou_sum += dqn_intersection / dqn_union
        
        # ========== Thin cloud IoU ==========
        baseline_thin_inter = np.sum(baseline_flat & gt_thin_flat)
        baseline_thin_union = np.sum(baseline_flat | gt_thin_flat)
        if baseline_thin_union > 0:
            baseline_thin_iou_sum += baseline_thin_inter / baseline_thin_union
        
        ppo_thin_inter = np.sum(ppo_flat & gt_thin_flat)
        ppo_thin_union = np.sum(ppo_flat | gt_thin_flat)
        if ppo_thin_union > 0:
            ppo_thin_iou_sum += ppo_thin_inter / ppo_thin_union
        
        dqn_thin_inter = np.sum(dqn_flat & gt_thin_flat)
        dqn_thin_union = np.sum(dqn_flat | gt_thin_flat)
        if dqn_thin_union > 0:
            dqn_thin_iou_sum += dqn_thin_inter / dqn_thin_union
        
        n_images += 1
        
    except Exception as e:
        print(f"Error on image {i}: {e}")
        continue

print(f"\n✅ Evaluated {n_images} images successfully!")

## 7. Compute and Display Final Metrics

In [ ]:
def compute_all_metrics(m):
    """Compute all metrics from confusion matrix components."""
    tp, fp, tn, fn = m['tp'], m['fp'], m['tn'], m['fn']
    total = tp + fp + tn + fn
    
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    
    return accuracy, precision, recall, f1, iou

# Compute metrics for all three models
b_acc, b_prec, b_rec, b_f1, b_iou = compute_all_metrics(baseline_metrics)
p_acc, p_prec, p_rec, p_f1, p_iou = compute_all_metrics(ppo_metrics)
d_acc, d_prec, d_rec, d_f1, d_iou = compute_all_metrics(dqn_metrics)

# Thin cloud recall
b_thin_recall = baseline_thin['tp'] / baseline_thin['total'] if baseline_thin['total'] > 0 else 0
p_thin_recall = ppo_thin['tp'] / ppo_thin['total'] if ppo_thin['total'] > 0 else 0
d_thin_recall = dqn_thin['tp'] / dqn_thin['total'] if dqn_thin['total'] > 0 else 0

# Average IoU across images
b_avg_iou = baseline_iou_sum / n_images if n_images > 0 else 0
p_avg_iou = ppo_iou_sum / n_images if n_images > 0 else 0
d_avg_iou = dqn_iou_sum / n_images if n_images > 0 else 0

b_avg_thin_iou = baseline_thin_iou_sum / n_images if n_images > 0 else 0
p_avg_thin_iou = ppo_thin_iou_sum / n_images if n_images > 0 else 0
d_avg_thin_iou = dqn_thin_iou_sum / n_images if n_images > 0 else 0

# Print results
print("\n" + "="*90)
print("📊 COMPLETE MODEL EVALUATION - 200 TEST IMAGES")
print("="*90)

print("\n📈 OVERALL CLOUD DETECTION METRICS:")
print("-"*90)
print(f"{'Metric':<25} {'CNN Baseline':>15} {'PPO (720k)':>15} {'DQN (100k)':>15} {'DQN vs CNN':>15}")
print("-"*90)
print(f"{'Accuracy':<25} {b_acc*100:>14.2f}% {p_acc*100:>14.2f}% {d_acc*100:>14.2f}% {(d_acc-b_acc)*100:>+14.2f}%")
print(f"{'Precision':<25} {b_prec*100:>14.2f}% {p_prec*100:>14.2f}% {d_prec*100:>14.2f}% {(d_prec-b_prec)*100:>+14.2f}%")
print(f"{'Recall':<25} {b_rec*100:>14.2f}% {p_rec*100:>14.2f}% {d_rec*100:>14.2f}% {(d_rec-b_rec)*100:>+14.2f}%")
print(f"{'F1-Score':<25} {b_f1*100:>14.2f}% {p_f1*100:>14.2f}% {d_f1*100:>14.2f}% {(d_f1-b_f1)*100:>+14.2f}%")
print(f"{'Overall IoU':<25} {b_iou*100:>14.2f}% {p_iou*100:>14.2f}% {d_iou*100:>14.2f}% {(d_iou-b_iou)*100:>+14.2f}%")
print(f"{'Average IoU (per image)':<25} {b_avg_iou*100:>14.2f}% {p_avg_iou*100:>14.2f}% {d_avg_iou*100:>14.2f}% {(d_avg_iou-b_avg_iou)*100:>+14.2f}%")

print("\n🌟 THIN CLOUD DETECTION (Primary Metric):")
print("-"*90)
print(f"{'Thin Cloud Recall':<25} {b_thin_recall*100:>14.2f}% {p_thin_recall*100:>14.2f}% {d_thin_recall*100:>14.2f}% {(d_thin_recall-b_thin_recall)*100:>+14.2f}%")
print(f"{'Thin Cloud IoU (avg)':<25} {b_avg_thin_iou*100:>14.2f}% {p_avg_thin_iou*100:>14.2f}% {d_avg_thin_iou*100:>14.2f}% {(d_avg_thin_iou-b_avg_thin_iou)*100:>+14.2f}%")

print("\n" + "="*90)
print("✅ EVALUATION COMPLETE!")
print("="*90)

## 8. Generate Thesis Figures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set up the style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16

# ============================================================
# FIGURE 4.3.1: Overall Metrics Comparison
# ============================================================
fig1, ax1 = plt.subplots(figsize=(12, 7))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'IoU']
x = np.arange(len(metrics))
width = 0.25

# Data
baseline_vals = [b_acc*100, b_prec*100, b_rec*100, b_f1*100, b_iou*100]
ppo_vals = [p_acc*100, p_prec*100, p_rec*100, p_f1*100, p_iou*100]
dqn_vals = [d_acc*100, d_prec*100, d_rec*100, d_f1*100, d_iou*100]

# Bars
bars1 = ax1.bar(x - width, baseline_vals, width, label='CNN Baseline', color='#3498db', edgecolor='black')
bars2 = ax1.bar(x, ppo_vals, width, label='PPO (720k steps)', color='#e74c3c', edgecolor='black')
bars3 = ax1.bar(x + width, dqn_vals, width, label='DQN (100k steps)', color='#2ecc71', edgecolor='black')

# Labels and formatting
ax1.set_ylabel('Percentage (%)')
ax1.set_title('Figure 4.3.1: Overall Performance Metrics Comparison\n(200 Test Images)', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.legend(loc='upper right')
ax1.set_ylim(0, 100)

# Add value labels on bars
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax1.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab_Data/Figure_4_3_1_Overall_Metrics.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure 4.3.1 saved!")

# ============================================================
# FIGURE 4.3.2: Thin Cloud Recall Comparison
# ============================================================
fig2, ax2 = plt.subplots(figsize=(10, 7))

models = ['CNN Baseline', 'PPO (720k steps)', 'DQN (100k steps)']
thin_recalls = [b_thin_recall*100, p_thin_recall*100, d_thin_recall*100]
colors = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax2.bar(models, thin_recalls, color=colors, edgecolor='black', width=0.6)

# Add value labels on top of bars
for bar, val in zip(bars, thin_recalls):
    ax2.annotate(f'{val:.2f}%',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=14, fontweight='bold')

# Add improvement annotations - positioned higher to avoid overlap
ppo_improvement = (p_thin_recall - b_thin_recall) * 100
dqn_improvement = (d_thin_recall - b_thin_recall) * 100

# Place improvement labels further above the value labels
ax2.annotate(f'(+{ppo_improvement:.2f}%)', 
            xy=(1, thin_recalls[1]), 
            xytext=(0, 25),  # Higher offset
            textcoords="offset points",
            ha='center', fontsize=10, color='#c0392b', fontweight='bold')
ax2.annotate(f'(+{dqn_improvement:.2f}%)', 
            xy=(2, thin_recalls[2]), 
            xytext=(0, 25),  # Higher offset
            textcoords="offset points",
            ha='center', fontsize=10, color='#27ae60', fontweight='bold')

ax2.set_ylabel('Thin Cloud Recall (%)')
ax2.set_title('Figure 4.3.2: Thin Cloud Recall Comparison\n(Primary Research Metric)', fontweight='bold')
ax2.set_ylim(0, 95)  # Adjusted to give room for labels

# Add horizontal line for baseline
ax2.axhline(y=thin_recalls[0], color='#3498db', linestyle='--', alpha=0.5, label='Baseline Level')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab_Data/Figure_4_3_2_Thin_Cloud_Recall.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure 4.3.2 saved!")

print("\n📁 Figures saved to Google Drive!")
print("   - Figure_4_3_1_Overall_Metrics.png")
print("   - Figure_4_3_2_Thin_Cloud_Recall.png")

## 8.5 Generate Methodology Pipeline Diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

# Colors
colors = {
    'input': '#3498db',      # Blue
    'cnn': '#9b59b6',        # Purple
    'rl': '#e74c3c',         # Red
    'output': '#2ecc71',     # Green
    'arrow': '#34495e',      # Dark gray
    'text': '#2c3e50'        # Dark text
}

def draw_box(ax, x, y, width, height, text, color, fontsize=11):
    """Draw a rounded rectangle with text."""
    box = FancyBboxPatch((x, y), width, height,
                         boxstyle="round,pad=0.05,rounding_size=0.3",
                         facecolor=color, edgecolor='black', linewidth=2,
                         alpha=0.85)
    ax.add_patch(box)
    ax.text(x + width/2, y + height/2, text,
            ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color='white', wrap=True)

def draw_arrow(ax, start, end, color='#34495e'):
    """Draw an arrow between two points."""
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))

# ============================================================
# TITLE
# ============================================================
ax.text(8, 9.5, 'Methodology Pipeline: CNN + RL Cloud Detection',
        ha='center', va='center', fontsize=18, fontweight='bold', color=colors['text'])

# ============================================================
# INPUT STAGE
# ============================================================
draw_box(ax, 0.5, 6.5, 3, 1.5, 'Sentinel-2 Image\n(13 Spectral Bands)', colors['input'])
ax.text(2, 6.2, '512 × 512 pixels', ha='center', fontsize=9, color='gray')

# ============================================================
# CNN BASELINE STAGE
# ============================================================
draw_box(ax, 5, 6.5, 3, 1.5, 'CNN Baseline\n(s2cloudless)', colors['cnn'])
ax.text(6.5, 6.2, 'LightGBM Ensemble', ha='center', fontsize=9, color='gray')

# Arrow: Input → CNN
draw_arrow(ax, (3.5, 7.25), (5, 7.25))

# ============================================================
# PROBABILITY MAP (intermediate)
# ============================================================
draw_box(ax, 9.5, 6.5, 3, 1.5, 'Cloud Probability\nMap [0, 1]', '#f39c12')
ax.text(11, 6.2, 'Threshold: 0.5', ha='center', fontsize=9, color='gray')

# Arrow: CNN → Probability
draw_arrow(ax, (8, 7.25), (9.5, 7.25))

# ============================================================
# RL REFINEMENT BRANCH (split into PPO and DQN)
# ============================================================
# PPO Branch
draw_box(ax, 4, 3.5, 3.5, 1.5, 'PPO Agent\n(Continuous Actions)', colors['rl'])
ax.text(5.75, 3.2, '720k training steps', ha='center', fontsize=9, color='gray')

# DQN Branch
draw_box(ax, 8.5, 3.5, 3.5, 1.5, 'DQN Agent\n(15 Discrete Actions)', colors['rl'])
ax.text(10.25, 3.2, '100k training steps', ha='center', fontsize=9, color='gray')

# Arrows: Probability → PPO and DQN
draw_arrow(ax, (10.5, 6.5), (5.75, 5))  # to PPO
draw_arrow(ax, (11.5, 6.5), (10.25, 5)) # to DQN

# ============================================================
# OBSERVATION & ACTION BOXES (side annotations)
# ============================================================
# Observation space
obs_box = FancyBboxPatch((0.3, 3.2), 2.8, 2,
                         boxstyle="round,pad=0.05,rounding_size=0.2",
                         facecolor='#ecf0f1', edgecolor='#7f8c8d', linewidth=1.5)
ax.add_patch(obs_box)
ax.text(1.7, 4.8, 'State (20 features)', ha='center', fontsize=10, fontweight='bold', color=colors['text'])
ax.text(1.7, 4.3, '• Prob statistics', ha='center', fontsize=8, color='gray')
ax.text(1.7, 3.9, '• Gradient/texture', ha='center', fontsize=8, color='gray')
ax.text(1.7, 3.5, '• Uncertainty ratio', ha='center', fontsize=8, color='gray')

# Reward function
reward_box = FancyBboxPatch((13, 3.2), 2.7, 2,
                            boxstyle="round,pad=0.05,rounding_size=0.2",
                            facecolor='#ecf0f1', edgecolor='#7f8c8d', linewidth=1.5)
ax.add_patch(reward_box)
ax.text(14.35, 4.8, 'Reward Function', ha='center', fontsize=10, fontweight='bold', color=colors['text'])
ax.text(14.35, 4.2, 'R = 0.7 × Thin IoU', ha='center', fontsize=9, color='gray')
ax.text(14.35, 3.7, '  + 0.3 × F1', ha='center', fontsize=9, color='gray')

# ============================================================
# OUTPUT STAGE
# ============================================================
# CNN Output
draw_box(ax, 1, 0.8, 3, 1.2, 'CNN Mask\n(Baseline)', colors['input'], fontsize=10)

# PPO Output
draw_box(ax, 5, 0.8, 3, 1.2, 'PPO Refined\nMask', colors['rl'], fontsize=10)

# DQN Output
draw_box(ax, 9, 0.8, 3, 1.2, 'DQN Refined\nMask', colors['output'], fontsize=10)

# Arrows to outputs
draw_arrow(ax, (11, 6.5), (2.5, 2))      # Prob → CNN output
draw_arrow(ax, (5.75, 3.5), (6.5, 2))    # PPO → PPO output
draw_arrow(ax, (10.25, 3.5), (10.5, 2))  # DQN → DQN output

# ============================================================
# RESULTS ANNOTATION
# ============================================================
results_box = FancyBboxPatch((12.5, 0.5), 3.3, 1.8,
                             boxstyle="round,pad=0.05,rounding_size=0.2",
                             facecolor='#d5f5e3', edgecolor='#27ae60', linewidth=2)
ax.add_patch(results_box)
ax.text(14.15, 1.9, '🎯 Thin Cloud Recall', ha='center', fontsize=10, fontweight='bold', color='#27ae60')
ax.text(14.15, 1.45, 'CNN: 53.72%', ha='center', fontsize=9, color='gray')
ax.text(14.15, 1.05, 'PPO: 62.88% (+9.17%)', ha='center', fontsize=9, color='#c0392b')
ax.text(14.15, 0.65, 'DQN: 71.32% (+17.60%)', ha='center', fontsize=9, fontweight='bold', color='#27ae60')

# ============================================================
# LEGEND
# ============================================================
legend_elements = [
    mpatches.Patch(facecolor=colors['input'], edgecolor='black', label='Input/CNN Baseline'),
    mpatches.Patch(facecolor=colors['cnn'], edgecolor='black', label='CNN Processing'),
    mpatches.Patch(facecolor=colors['rl'], edgecolor='black', label='RL Agents'),
    mpatches.Patch(facecolor=colors['output'], edgecolor='black', label='Best Output (DQN)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab_Data/Figure_3_1_Methodology_Pipeline.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Pipeline diagram saved as Figure_3_1_Methodology_Pipeline.png")

## 8.6 Generate Pipeline Images for Canva

In [ ]:
# ============================================================
# Generate all pipeline images for Canva
# ============================================================
import os
import matplotlib.pyplot as plt
import numpy as np

# Create output directory
save_dir = '/content/drive/MyDrive/Colab_Data/pipeline_images'
os.makedirs(save_dir, exist_ok=True)

# Pick a good sample image (change index to try different images: 0, 10, 50, etc.)
sample_idx = 10

img_path = test_images[sample_idx]
mask_path = test_masks[sample_idx]

print(f"Using sample image: {os.path.basename(img_path)}")

# Load image and ground truth
with rasterio.open(img_path) as src:
    bands = src.read()
with rasterio.open(mask_path) as src:
    gt = src.read(1)

# Get CNN probability
cnn_prob = get_cnn_probability(img_path)
baseline_pred = (cnn_prob > 0.5).astype(np.uint8)

# Get DQN refined prediction
env_dqn = ThinCloudDetectionEnvDiscrete(cnn_prob, gt)
obs, _ = env_dqn.reset()
done = False
while not done:
    action, _ = dqn_model.predict(obs, deterministic=True)
    obs, _, done, _, _ = env_dqn.step(action)
dqn_pred = env_dqn.get_refined_mask()
dqn_refined_prob = env_dqn.refined_prob  # Get the refined probability map

# ============================================================
# IMAGE 1: Original Satellite Image (RGB)
# ============================================================
rgb = np.stack([bands[3], bands[2], bands[1]], axis=-1)  # B4, B3, B2
rgb = np.clip(rgb / 3000, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.axis('off')
plt.savefig(f'{save_dir}/1_original_rgb.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 1_original_rgb.png saved")

# ============================================================
# IMAGE 2: CNN Probability Map (grayscale 0-1)
# ============================================================
plt.figure(figsize=(8, 8))
plt.imshow(cnn_prob, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.savefig(f'{save_dir}/2_probability_map.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 2_probability_map.png saved")

# With colorbar version
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(cnn_prob, cmap='RdYlBu_r', vmin=0, vmax=1)
ax.axis('off')
cbar = plt.colorbar(im, ax=ax, shrink=0.8, label='Cloud Probability')
plt.savefig(f'{save_dir}/2b_probability_map_colorbar.png', dpi=200, bbox_inches='tight')
plt.close()
print("✅ 2b_probability_map_colorbar.png saved")

# ============================================================
# IMAGE 3: DQN Refined Probability Map
# ============================================================
plt.figure(figsize=(8, 8))
plt.imshow(dqn_refined_prob, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.savefig(f'{save_dir}/3_refined_probability.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 3_refined_probability.png saved")

# With colorbar version
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(dqn_refined_prob, cmap='RdYlBu_r', vmin=0, vmax=1)
ax.axis('off')
cbar = plt.colorbar(im, ax=ax, shrink=0.8, label='Refined Probability')
plt.savefig(f'{save_dir}/3b_refined_probability_colorbar.png', dpi=200, bbox_inches='tight')
plt.close()
print("✅ 3b_refined_probability_colorbar.png saved")

# ============================================================
# IMAGE 4: Final Binary Mask (CNN Baseline)
# ============================================================
plt.figure(figsize=(8, 8))
plt.imshow(baseline_pred, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.savefig(f'{save_dir}/4_cnn_binary_mask.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 4_cnn_binary_mask.png saved")

# ============================================================
# IMAGE 5: Final Binary Mask (DQN Refined)
# ============================================================
plt.figure(figsize=(8, 8))
plt.imshow(dqn_pred, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.savefig(f'{save_dir}/5_dqn_binary_mask.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 5_dqn_binary_mask.png saved")

# ============================================================
# IMAGE 6: Ground Truth (3-class colored)
# ============================================================
colored_gt = np.zeros((*gt.shape, 3))
colored_gt[gt == 0] = [0.2, 0.7, 0.3]   # Clear - Green
colored_gt[gt == 1] = [0.9, 0.2, 0.2]   # Thick cloud - Red
colored_gt[gt == 2] = [1.0, 0.85, 0.4]  # Thin cloud - Yellow

plt.figure(figsize=(8, 8))
plt.imshow(colored_gt)
plt.axis('off')
plt.savefig(f'{save_dir}/6_ground_truth_colored.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 6_ground_truth_colored.png saved")

# ============================================================
# IMAGE 7: Comparison - What DQN found that CNN missed
# ============================================================
gt_thin = (gt == 2)
comparison = np.zeros((*dqn_pred.shape, 3))
comparison[(dqn_pred == 1) & (baseline_pred == 1)] = [0.5, 0.5, 0.5]  # Both detected - Gray
comparison[(dqn_pred == 1) & (baseline_pred == 0)] = [0.2, 0.9, 0.3]  # DQN found - Green
comparison[(baseline_pred == 1) & (dqn_pred == 0)] = [0.9, 0.3, 0.3]  # CNN only - Red

plt.figure(figsize=(8, 8))
plt.imshow(comparison)
plt.axis('off')
plt.savefig(f'{save_dir}/7_improvement_comparison.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 7_improvement_comparison.png saved")

# ============================================================
# IMAGE 8: Patch Grid Visualization
# ============================================================
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb)
for i in range(0, 512, 64):
    ax.axhline(y=i, color='white', linewidth=1, alpha=0.8)
    ax.axvline(x=i, color='white', linewidth=1, alpha=0.8)
ax.axis('off')
plt.savefig(f'{save_dir}/8_patch_grid.png', dpi=200, bbox_inches='tight', pad_inches=0)
plt.close()
print("✅ 8_patch_grid.png saved")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("📁 ALL PIPELINE IMAGES SAVED!")
print("="*60)
print(f"\nLocation: {save_dir}")
print("\nFiles for Canva:")
print("  1_original_rgb.png           - Satellite image (RGB)")
print("  2_probability_map.png        - CNN probability (grayscale)")
print("  2b_probability_map_colorbar.png - With colorbar")
print("  3_refined_probability.png    - DQN refined probability")
print("  3b_refined_probability_colorbar.png - With colorbar")
print("  4_cnn_binary_mask.png        - CNN final mask (B/W)")
print("  5_dqn_binary_mask.png        - DQN final mask (B/W)")
print("  6_ground_truth_colored.png   - Ground truth (colored)")
print("  7_improvement_comparison.png - What DQN found (green)")
print("  8_patch_grid.png             - 64x64 patch visualization")
print("\n💡 Tip: Try different sample_idx values (0-199) for better examples!")

## 9. Summary Table for Thesis

In [ ]:
print("\n📋 COPY THIS TO YOUR THESIS:")
print("="*80)

print("\n### Table 4.1.1: CNN Baseline Performance")
print(f"| Metric | Value |")
print(f"|--------|-------|")
print(f"| Overall Accuracy | {b_acc*100:.2f}% |")
print(f"| Precision | {b_prec*100:.2f}% |")
print(f"| Overall Recall | {b_rec*100:.2f}% |")
print(f"| F1-Score | {b_f1*100:.2f}% |")
print(f"| Overall IoU | {b_iou*100:.2f}% |")
print(f"| Thin Cloud Recall | {b_thin_recall*100:.2f}% |")

print("\n### Table 4.3.1: Overall Performance Metrics Comparison")
print(f"| Model | Accuracy | Precision | Recall | F1-Score | IoU |")
print(f"|-------|----------|-----------|--------|----------|-----|")
print(f"| CNN Baseline | {b_acc*100:.2f}% | {b_prec*100:.2f}% | {b_rec*100:.2f}% | {b_f1*100:.2f}% | {b_iou*100:.2f}% |")
print(f"| PPO (720k steps) | {p_acc*100:.2f}% | {p_prec*100:.2f}% | {p_rec*100:.2f}% | {p_f1*100:.2f}% | {p_iou*100:.2f}% |")
print(f"| DQN (100k steps) | {d_acc*100:.2f}% | {d_prec*100:.2f}% | {d_rec*100:.2f}% | {d_f1*100:.2f}% | {d_iou*100:.2f}% |")

print("\n### Table 4.3.2: Thin Cloud Recall Comparison")
print(f"| Model | Thin Cloud Recall | Improvement vs. Baseline |")
print(f"|-------|-------------------|--------------------------|")
print(f"| CNN Baseline | {b_thin_recall*100:.2f}% | — |")
print(f"| PPO (720k steps) | {p_thin_recall*100:.2f}% | {(p_thin_recall-b_thin_recall)*100:+.2f}% |")
print(f"| DQN (100k steps) | {d_thin_recall*100:.2f}% | {(d_thin_recall-b_thin_recall)*100:+.2f}% |")

print("\n### Key Findings Summary:")
print(f"- DQN Thin Cloud Recall Improvement: {(d_thin_recall-b_thin_recall)*100:+.2f}%")
print(f"- PPO Thin Cloud Recall Improvement: {(p_thin_recall-b_thin_recall)*100:+.2f}%")
print(f"- DQN IoU Improvement: {(d_iou-b_iou)*100:+.2f}%")
print(f"- PPO IoU Improvement: {(p_iou-b_iou)*100:+.2f}%")
print(f"- DQN vs PPO (Thin Cloud): {(d_thin_recall-p_thin_recall)*100:+.2f}%")

print("\n" + "="*80)